Axl Waltz Cruz<br>
02/24/26<br>
CPE301-CPE22S3<br>
Engr. Neal Matira

# **Data Gathering**

# **Example of gathering image data using webcam**

In [11]:
import cv2

# Start webcam
webcam = cv2.VideoCapture(0)

print("Press 's' to capture image")
print("Press 'q' to quit")

while True:
    try:
        # Read frame
        check, frame = webcam.read()
        print(check)  # True if webcam works

        if not check:
            print("Failed to access webcam")
            break

        # Show live camera
        cv2.imshow("Capturing", frame)

        key = cv2.waitKey(1)

        # Press 's' to save image
        if key == ord('s'):
            cv2.imwrite('saved_img.jpg', frame)
            print("Image saved!")

            # Convert to grayscale
            print("Converting RGB image to grayscale...")
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

            # Resize to 28x28
            print("Resizing image to 28x28...")
            resized = cv2.resize(gray, (28, 28))

            # Save final processed image
            cv2.imwrite('saved_img-final.jpg', resized)
            print("Processed image saved!")

            # Show grayscale image
            cv2.imshow("Captured Image", resized)
            cv2.waitKey(1650)

            break

        # Press 'q' to quit
        elif key == ord('q'):
            print("Turning off camera.")
            break

    except KeyboardInterrupt:
        print("Interrupted by user.")
        break

# Release resources
webcam.release()
cv2.destroyAllWindows()
print("Camera off. Program ended.")

Press 's' to capture image
Press 'q' to quit
False
Failed to access webcam
Camera off. Program ended.


# **Example of gathering voice data using microphone**

In [3]:
!pip3 install sounddevice

In [4]:
!pip3 install wavio

In [5]:
!pip3 install scipy

In [6]:
!apt-get install libportaudio2

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  libportaudio2
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 65.3 kB of archives.
After this operation, 223 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libportaudio2 amd64 19.6.0-1.1 [65.3 kB]
Fetched 65.3 kB in 1s (109 kB/s)
Selecting previously unselected package libportaudio2:amd64.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../libportaudio2_19.6.0-1.1_amd64.deb ...
Unpacking libportaudio2:amd64 (19.6.0-1.1) ...
Setting up libportaudio2:amd64 (19.6.0-1.1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.11) ...
/sbin/ldconfig.real: /usr/local/lib/libtbbmalloc.so.2 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/l

In [9]:
# Import required libraries
import sounddevice as sd
from scipy.io.wavfile import write
import wavio
import numpy as np

# Sampling frequency (Hz)
freq = 44100

# Recording duration (seconds)
duration = 5

print("Recording started...")

# Start recording (stereo: channels=2)
recording = sd.rec(int(duration * freq),
                   samplerate=freq,
                   channels=2)

# Wait until recording is finished
sd.wait()

print("Recording finished.")

# Convert float data to int16 (required for scipy write)
recording_int16 = np.int16(recording * 32767)

# Save using scipy
write("recording0.wav", freq, recording_int16)

# Save using wavio
wavio.write("recording1.wav", recording, freq, sampwidth=2)

print("Audio files saved successfully!")

Recording started...


PortAudioError: Error querying device -1

# **Web Scraping**

In [12]:
!pip install bs4

In [13]:
pip install requests

In [15]:
import requests
from bs4 import BeautifulSoup

def getdata(url):
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    r = requests.get(url, headers=headers)
    return r.text

# Get HTML data
htmldata = getdata("https://www.google.com")

# Parse HTML
soup = BeautifulSoup(htmldata, "html.parser")

# Extract all image sources
for item in soup.find_all("img"):
    src = item.get("src")   # safer than item['src']
    if src:
        print(src)

/images/branding/googlelogo/1x/googlelogo_white_background_color_272x92dp.png


In [1]:
pip install selenium


# **Image Scraping using Selenium**

In [3]:
!pip install selenium
!apt-get update
!apt install -y chromium-chromedriver
!cp /usr/lib/chromium-browser/chromedriver /usr/bin
import time
import requests
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# Setup Chrome options
chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Start driver
driver = webdriver.Chrome(options=chrome_options)

# Function to scroll page
def scroll_to_end(driver):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

# Function to get image URLs
def get_image_urls(query, total_imgs):
    search_url = f"https://www.google.com/search?q={query}&tbm=isch"
    driver.get(search_url)

    img_urls = set()

    while len(img_urls) < total_imgs:
        scroll_to_end(driver)

        thumbnails = driver.find_elements(By.XPATH, "//img[contains(@class,'Q4LuWd')]")

        for img in thumbnails:
            try:
                img.click()
                time.sleep(1)

                images = driver.find_elements(By.CSS_SELECTOR, "img.n3VNCb")
                for image in images:
                    src = image.get_attribute("src")
                    if src and "http" in src:
                        img_urls.add(src)

                if len(img_urls) >= total_imgs:
                    break

            except:
                continue

        print(f"Collected: {len(img_urls)} images")

    return list(img_urls)[:total_imgs]

# Function to download images
def download_images(folder, urls):
    if not os.path.exists(folder):
        os.makedirs(folder)

    for i, url in enumerate(urls):
        try:
            image_content = requests.get(url).content
            with open(os.path.join(folder, f"image_{i}.jpg"), "wb") as f:
                f.write(image_content)
        except Exception as e:
            print(f"Could not download {url} - {e}")

# 🔹 Run scraper
urls = get_image_urls("Car", 10)
download_images("car_images", urls)

driver.quit()
print("Done!")

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
chromium-chromedriver is already the newest version (1:85.0.4183.83-0ubuntu2.22.04.1).
0 upgra

SessionNotCreatedException: Message: session not created: Chrome instance exited. Examine ChromeDriver verbose log to determine the cause.; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#sessionnotcreatedexception
Stacktrace:
#0 0x599195b41b6a <unknown>
#1 0x599195554a32 <unknown>
#2 0x5991955925bb <unknown>
#3 0x59919558d289 <unknown>
#4 0x5991955dec2e <unknown>
#5 0x5991955de31c <unknown>
#6 0x59919559bc0f <unknown>
#7 0x59919559c9d1 <unknown>
#8 0x599195b066b9 <unknown>
#9 0x599195b095c1 <unknown>
#10 0x599195af2e29 <unknown>
#11 0x599195b0a17e <unknown>
#12 0x599195ad94b0 <unknown>
#13 0x599195b2e578 <unknown>
#14 0x599195b2e74b <unknown>
#15 0x599195b401a3 <unknown>
#16 0x7e6e5ddb3ac3 <unknown>
